## Gold — `fato_populacao` (star schema estimativa populacional)

**Origem:** `workspace.silver.populacao_estimada` + `gold.dim_municipio` → **Destino:** `workspace.gold.fato_populacao`

- **Modelo:** Star Schema. Fato de **snapshot periódico** — grão: **1 linha por município x ano** (1991-2025).
- **Dimensões:** `dim_municipio` (geografia, via `sk_municipio`); `ano` fica como dimensão degenerada (o calendário `dim_data` é diário e não se aplica ao grão anual).
- **Medida:** `populacao`.
- **Transformações:**
  - Lookup da SK: join por `codigo_municipio` (IBGE 7 dígitos) contra `gold.dim_municipio.codigo_municipio` — a silver já validou a existência do registro.
  - Ordem das colunas: SK, natural key (`codigo_municipio`), ano e medida.
- **Linhagem:** Base dos Dados CSV → `bronze.estimativa_populacional` → `silver.populacao_estimada` (+ `silver.municipios`) → `gold.dim_municipio` → `gold.fato_populacao`.

In [0]:
%run ../shared/_setup

In [0]:
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments, add_table_comment
from catalogo.populacao import FATO_POPULACAO_COMMENTS, FATO_POPULACAO_TABLE_COMMENT

In [0]:
SILVER_POPULACAO = "workspace.silver.populacao_estimada"
DIM_MUNICIPIO = "workspace.gold.dim_municipio"
TARGET_TABLE = "workspace.gold.fato_populacao"

# Contrato de saída gold.fato_populacao
COLUNAS_ORDENADAS = [
    "sk_municipio",
    "codigo_municipio",
    "ano",
    "populacao",
]

In [0]:
df_silver = spark.table(SILVER_POPULACAO)
print(f"Silver populacao_estimada: {df_silver.count():,} linhas | anos: {df_silver.select('ano').distinct().count()}")

# dim_municipio: código IBGE -> SK (a silver já validou o registro)
df_municipio = spark.table(DIM_MUNICIPIO).select(
    F.col("codigo_municipio"),
    F.col("sk_municipio"),
)
df = df_silver.join(df_municipio, on="codigo_municipio", how="left")

df = df.select(*COLUNAS_ORDENADAS)
print(f"Fato final: {df.count():,} linhas")
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    FATO_POPULACAO_COMMENTS
)
add_table_comment(spark, TARGET_TABLE, FATO_POPULACAO_TABLE_COMMENT)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
pk_distintos = spark.table(TARGET_TABLE).select("codigo_municipio", "ano").distinct().count()
sk_nulas = spark.table(TARGET_TABLE).filter(F.col("sk_municipio").isNull()).count()
print(f"Total: {total:,} | PK distintos (município x ano): {pk_distintos:,} | FK nulas: {sk_nulas}")
assert total == pk_distintos and sk_nulas == 0, "Quebra de grão/FK em fato_populacao"
display(spark.sql(f"SELECT ano, count(*) AS qtd_municipios, sum(populacao) AS populacao_brasil FROM {TARGET_TABLE} GROUP BY ano ORDER BY ano"))
display(spark.sql(
    f"SELECT d.sigla_uf, sum(f.populacao) AS populacao_2025 \
      FROM {TARGET_TABLE} f JOIN {DIM_MUNICIPIO} d ON f.sk_municipio = d.sk_municipio \
      WHERE f.ano = 2025 GROUP BY d.sigla_uf ORDER BY populacao_2025 DESC"
))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))